# Tugas 6 :  Sistem pencarian Query dengan kalimat dalam dokumen dengan penerapan Singular Value Decomposition (SVD)

NAMA : Mohammad Iqbal Surya Ramadhan

NIM  : 210411100002

MATA KULIAH : Pencarian dan Penambangan Web - A

mencari 100 dokumen itu (masukkan 1 kalimat, ketemunya dokumen yg mana)

data tfidf ditransofrmasi menjadi sdikit dimensi menggunakan svd
yg diambil v dan sigma (v tdk diambil smua)(Sebagian dri v)
mentransformasi supaya dikit kolom


cosinus similtas (mirip mendekati 1) boleh pke euclidean

- membuat sistem pencarian dokumen
- reduksi dimensi SVD
- data baru juga di reduksi
- mencari kemiripan ecludian distance

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import pandas as pd

data_path = ("/content/drive/My Drive/PPWA/report/Tugas-PPWA/Hasil_Prepros.csv")
news_data = pd.read_csv(data_path)

news_data

,judul,tanggal,isi,kategori,cleansing,case_folding,tokenize,stopword_removal
0,"Nunggak 8 Bulan, Segini Pajak Ford Mustang Mil...","Kamis, 19 Sep 2024 13:05 WIB",Jakarta - Bareskrim Polri menyita aset senilai...,Otomotif,Jakarta Bareskrim Polri menyita aset senilai ...,jakarta bareskrim polri menyita aset senilai ...,"['jakarta', 'bareskrim', 'polri', 'menyita', '...",jakarta bareskrim polri menyita aset senilai r...
1,Bos Ford Kaget usai Jajal Mobil China: Mereka ...,"Kamis, 19 Sep 2024 12:33 WIB","Jakarta - Chief Executive Officer (CEO) Ford, ...",Otomotif,Jakarta Chief Executive Officer CEO Ford Jim ...,jakarta chief executive officer ceo ford jim ...,"['jakarta', 'chief', 'executive', 'officer', '...",jakarta chief executive officer ceo ford jim f...
2,"Tarif Tol Dalam Kota Naik Jadi Segini, Berlaku...","Kamis, 19 Sep 2024 12:08 WIB",Jakarta - Jasa Marga mengumumkan kenaikan tari...,Otomotif,Jakarta Jasa Marga mengumumkan kenaikan tarif...,jakarta jasa marga mengumumkan kenaikan tarif...,"['jakarta', 'jasa', 'marga', 'mengumumkan', 'k...",jakarta jasa marga mengumumkan kenaikan tarif ...
3,Pak RT Aleix Espargaro Tak Sabar Balapan Terak...,"Kamis, 19 Sep 2024 11:40 WIB","Jakarta - Pebalap Aprilia asal Spanyol, Aleix ...",Otomotif,Jakarta Pebalap Aprilia asal Spanyol Aleix Es...,jakarta pebalap aprilia asal spanyol aleix es...,"['jakarta', 'pebalap', 'aprilia', 'asal', 'spa...",jakarta pebalap aprilia spanyol aleix espargar...
4,Angkot Listrik Bakal Diuji Coba di Jakarta,"Kamis, 19 Sep 2024 11:18 WIB",Jakarta - PT Transportasi Jakarta (TransJakart...,Otomotif,Jakarta PT Transportasi Jakarta TransJakarta ...,jakarta pt transportasi jakarta transjakarta ...,"['jakarta', 'pt', 'transportasi', 'jakarta', '...",jakarta pt transportasi jakarta transjakarta k...
...,...,...,...,...,...,...,...,...
95,Suku Bunga BI Dipangkas Jadi 6% Dinilai Berani,"Rabu, 18 Sep 2024 21:15 WIB",Jakarta - Bank Indonesia (BI) memutuskan untuk...,Keuangan,Jakarta Bank Indonesia BI memutuskan untuk me...,jakarta bank indonesia bi memutuskan untuk me...,"['jakarta', 'bank', 'indonesia', 'bi', 'memutu...",jakarta bank indonesia bi memutuskan menurunka...
96,Prabowo Ingatkan Perang Makin Marak hingga Sal...,"Rabu, 18 Sep 2024 21:00 WIB",Jakarta - Presiden terpilih Prabowo Subianto m...,Keuangan,Jakarta Presiden terpilih Prabowo Subianto me...,jakarta presiden terpilih prabowo subianto me...,"['jakarta', 'presiden', 'terpilih', 'prabowo',...",jakarta presiden terpilih prabowo subianto men...
97,"Ditjen Pajak Respons Kabar 6 Juta NPWP Bocor, ...","Rabu, 18 Sep 2024 20:36 WIB",Jakarta - Direktorat Jenderal Pajak (DJP) Keme...,Keuangan,Jakarta Direktorat Jenderal Pajak DJP Kemente...,jakarta direktorat jenderal pajak djp kemente...,"['jakarta', 'direktorat', 'jenderal', 'pajak',...",jakarta direktorat jenderal pajak djp kementer...
98,Buruh Waswas Kisruh di Kadin Pengaruhi Penetap...,"Rabu, 18 Sep 2024 20:30 WIB",Jakarta - Presiden Partai Buruh yang juga Pres...,Keuangan,Jakarta Presiden Partai Buruh yang juga Presi...,jakarta presiden partai buruh yang juga presi...,"['jakarta', 'presiden', 'partai', 'buruh', 'ya...",jakarta presiden partai buruh presiden konfede...


In [29]:
# Ambil kolom dokumen berita
documents = news_data['stopword_removal'].tolist()

**TF-IDF (Term Frequency-Inverse Document Frequency)**

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import euclidean_distances
import pandas as pd
import numpy as np
import joblib

*   TfidfVectorizer adalah alat dari pustaka scikit-learn untuk mengubah kumpulan teks menjadi representasi vektor berbasis Term Frequency-Inverse Document Frequency (TF-IDF)

*   from sklearn.feature_extraction.text adalah modul dalam pustaka scikit-learn yang menyediakan berbagai alat untuk mengekstraksi fitur atau informasi numerik dari data teks (seperti dokumen atau kalimat) agar bisa digunakan dalam model pembelajaran mesin. Salah satu alat penting yang disediakan modul ini adalah TfidfVectorizer, yang digunakan untuk mengonversi teks menjadi representasi numerik berbasis frekuensi.

*   from sklearn.decomposition import TruncatedSVD adalah bagian dari pustaka scikit-learn yang menyediakan alat untuk melakukan reduksi dimensi pada data numerik, terutama matriks besar, dengan menggunakan teknik Truncated Singular Value Decomposition (Truncated SVD). Truncated SVD biasanya diterapkan pada data yang telah diproses menjadi bentuk matriks dengan representasi TF-IDF atau lainnya, terutama pada data teks untuk analisis Latent Semantic Analysis (LSA).

*   Truncated SVD adalah varian dari Singular Value Decomposition (SVD), yang menguraikan matriks menjadi tiga matriks komponen, tetapi dengan hanya menyimpan komponen terbesar. Dalam konteks pembelajaran mesin dan pemrosesan data teks, tujuan utamanya adalah untuk mereduksi dimensi data agar menjadi lebih sederhana, namun tetap mempertahankan informasi penting.

*   from sklearn.metrics.pairwise import euclidean_distances adalah bagian dari pustaka scikit-learn yang digunakan untuk menghitung jarak Euclidean antara pasangan data dalam matriks. Jarak Euclidean adalah ukuran jarak "lurus" antara dua titik dalam ruang dimensi. Di dalam konteks pemrosesan teks atau data numerik, ini sering digunakan untuk mengukur seberapa dekat atau jauh dua dokumen atau dua sampel dalam ruang vektor.

*   euclidean_distances digunakan untuk menghitung jarak Euclidean antara setiap pasangan baris dalam dua matriks. Jika hanya ada satu matriks yang diberikan, euclidean_distances akan menghitung jarak Euclidean antara semua baris dalam matriks tersebut, sehingga menghasilkan matriks jarak.

Matriks jarak Euclidean menunjukkan jarak antara setiap dokumen:

*   Nilai yang lebih kecil mengindikasikan bahwa dua dokumen tersebut lebih mirip secara semantik.

*   Nilai yang lebih besar menunjukkan bahwa dua dokumen memiliki perbedaan yang lebih jauh.

*   pandas adalah pustaka Python yang digunakan untuk manipulasi dan analisis data. Dengan menggunakan pandas, Anda dapat dengan mudah bekerja dengan data yang terstruktur dalam bentuk tabel (DataFrame) serta melakukan operasi seperti pemfilteran, pengelompokan, agregasi, dan transformasi data.

*   numpy adalah pustaka Python yang digunakan untuk komputasi numerik. Pustaka ini menyediakan dukungan untuk array multidimensi dan berbagai fungsi matematis yang efisien untuk memanipulasi data tersebut.

*   Digunakan untuk menyimpan dan memuat objek Python, joblib adalah pustaka Python yang digunakan untuk serialisasi (menyimpan) dan deserialisasi (memuat) objek Python, khususnya yang besar, seperti model machine learning.















In [31]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

# Mendapatkan fitur (kata-kata) dari TF-IDF
tfidf_features = tfidf_vectorizer.get_feature_names_out()

# Mengonversi matriks TF-IDF menjadi DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_features)

# Tampilkan hasil TF-IDF
print("\nHasil TF-IDF dalam bentuk DataFrame:")
tfidf_df


Hasil TF-IDF dalam bentuk DataFrame:


,abang,abdullah,abs,absen,ac,acara,acaraacara,acc,accord,acdkil,...,zenix,zero,zhejiang,ziko,zimbabwe,zona,zoning,zs,zulkifli,zz
0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
1,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.0,0.0,0.0,0.114233,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
96,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
97,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.078399,0.0
98,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0



1.   membuat sebuah objek TfidfVectorizer. Ini adalah langkah pertama untuk mengonversi kumpulan dokumen teks menjadi representasi numerik berdasarkan frekuensi kata dengan mempertimbangkan seberapa umum atau jarang kata tersebut dalam koleksi dokumen.

2.   TfidfVectorizer mengubah teks menjadi matriks TF-IDF, yang merupakan representasi numerik dari kata-kata dalam dokumen.

3.   menggunakan metode fit_transform() pada objek tfidf_vectorizer untuk menerapkan TF-IDF ke kumpulan dokumen yang disimpan dalam variabel documents.

4.   Hasilnya, tfidf_matrix adalah matriks yang menyimpan nilai TF-IDF untuk setiap kata dalam setiap dokumen. Setiap baris mewakili satu dokumen, dan setiap kolom mewakili satu kata (fitur) dari kumpulan dokumen.

5.   Dengan menggunakan get_feature_names_out(), kita dapat mengambil daftar kata (fitur) yang telah diidentifikasi oleh TfidfVectorizer. Ini adalah kata-kata unik yang ada dalam kumpulan dokumen.

6.   Variabel tfidf_features sekarang berisi daftar semua kata yang digunakan dalam analisis TF-IDF.

7.   mengonversi matriks TF-IDF (yang awalnya dalam bentuk sparse matrix) menjadi DataFrame menggunakan pd.DataFrame().

8.   Metode toarray() digunakan untuk mengubah sparse matrix menjadi array NumPy, yang kemudian dapat dengan mudah dimasukkan ke dalam DataFrame.

9.   columns=tfidf_features memberikan nama kolom pada DataFrame sesuai dengan kata-kata (fitur) yang ada.

10.    menampilkan DataFrame tfidf_df yang berisi nilai TF-IDF untuk setiap kata di setiap dokumen.










**Singular Value Decomposition (SVD)**

In [32]:
# Menggunakan SVD untuk reduksi dimensi
n_components = 100
svd = TruncatedSVD(n_components=n_components)
svd_matrix = svd.fit_transform(tfidf_matrix)

# Mengubah hasil SVD menjadi DataFrame
svd_df = pd.DataFrame(svd_matrix, columns=[f'feature {i+1}' for i in range(n_components)])

# Menampilkan hasil
print("Hasil reduksi dimensi dengan SVD:")
svd_df

Hasil reduksi dimensi dengan SVD:


,feature 1,feature 2,feature 3,feature 4,feature 5,feature 6,feature 7,feature 8,feature 9,feature 10,...,feature 91,feature 92,feature 93,feature 94,feature 95,feature 96,feature 97,feature 98,feature 99,feature 100
0,0.316538,-0.134168,-0.027742,-0.068346,0.032257,-0.037286,0.093471,0.258179,-0.027594,-0.032902,...,-0.002018,-0.033002,-0.006155,-0.021286,0.011730,0.000611,-0.002164,-0.011582,0.001889,0.000812
1,0.231510,-0.131316,-0.092982,0.137206,-0.001500,0.012837,0.014578,0.151100,0.060886,-0.046418,...,0.005756,0.000824,0.015479,0.004884,-0.015367,0.011901,0.002442,-0.000014,-0.000725,-0.000748
2,0.221785,0.019340,0.224015,-0.131731,0.618516,0.553204,0.167706,-0.141715,0.054393,-0.037489,...,0.012262,-0.002920,-0.003075,-0.003878,0.010500,0.009039,0.004980,-0.000812,0.045954,0.176552
3,0.120050,-0.031297,0.347339,0.103173,-0.236697,0.080723,0.003682,-0.095835,-0.027000,-0.053518,...,-0.023483,-0.003203,0.008976,-0.000348,0.004533,-0.000595,0.009531,0.000037,0.001314,-0.001483
4,0.141949,-0.042113,-0.002574,0.086886,0.034181,0.021903,-0.108858,0.087057,0.036474,-0.040637,...,-0.001531,-0.000369,-0.014043,-0.008642,0.000920,-0.006803,0.001787,-0.000661,-0.000705,0.001094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.253979,0.615201,-0.141088,-0.002460,-0.137124,0.081352,0.093642,0.015520,-0.033857,0.058014,...,0.039634,-0.072717,0.020862,-0.021014,0.060300,0.038089,0.003063,0.000433,-0.002443,-0.000326
96,0.134244,0.040500,0.052717,0.058221,0.080514,-0.123454,-0.094995,-0.060807,0.034049,-0.064869,...,-0.001463,0.013231,0.012004,0.009068,-0.002444,-0.004308,-0.001379,-0.001922,0.001588,0.000008
97,0.267603,-0.052670,0.167202,-0.270071,0.125769,-0.405008,0.155715,-0.173776,-0.296645,0.150403,...,0.004413,0.002951,-0.005031,0.001398,0.013459,-0.036855,0.001530,-0.308102,0.002074,0.000282
98,0.069874,0.030880,0.017691,0.008420,0.029645,-0.044422,-0.045202,-0.031068,0.010769,-0.031522,...,-0.003466,-0.004614,0.002227,0.000221,0.000430,0.001409,0.000314,0.002692,-0.000407,-0.000714


*   menggunakan SVD (Singular Value Decomposition) untuk reduksi dimensi pada matriks TF-IDF.

*   n_components: Di sini, kita menentukan jumlah komponen utama yang ingin kita ambil dari matriks TF-IDF. Dalam hal ini, kita ingin mengurangi dimensi ke 100 komponen. Jumlah ini bisa disesuaikan tergantung pada data dan tujuan analisis.

*   TruncatedSVD: Ini adalah kelas dari pustaka sklearn.decomposition yang digunakan untuk melakukan SVD dengan cara yang efisien pada data besar dan sparse (seperti matriks TF-IDF). TruncatedSVD melakukan reduksi dimensi dengan cara menyimpan informasi penting dari data dengan mengurangi jumlah dimensi.

*   fit_transform(): Metode ini melakukan dua langkah :

1.   Fit: Menyesuaikan model SVD ke matriks TF-IDF untuk menemukan struktur dan pola.

2.   Transform: Mengubah matriks TF-IDF menjadi bentuk baru (matriks SVD) dengan dimensi yang lebih rendah.

*   svd_matrix: Hasil dari fit_transform() adalah matriks baru yang memiliki dimensi n_samples x n_components. Setiap baris di svd_matrix merepresentasikan dokumen dalam ruang dimensi yang lebih rendah.

*   mengonversi hasil matriks SVD (yang merupakan array NumPy) menjadi sebuah DataFrame menggunakan pandas.

*   pd.DataFrame(svd_matrix, ...): Ini menciptakan DataFrame baru dengan baris yang sama dengan jumlah dokumen dan kolom yang sesuai dengan jumlah komponen yang telah ditentukan (100).

*   daftar komprehensif yang membuat nama kolom sebagai feature 1, feature 2, ..., hingga feature 100. Ini memberikan label pada kolom-kolom baru berdasarkan jumlah komponen.







**Implementasi**

In [33]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

*   import re: Memungkinkan kita menggunakan modul reguler ekspresi (regex) untuk pencarian dan manipulasi string. Ini berguna untuk membersihkan teks, seperti menghapus karakter yang tidak diinginkan.

*   import string: Digunakan untuk mengakses sekumpulan konstanta string, seperti daftar karakter alfabet, digit, dan tanda baca. Ini berguna untuk memfilter atau menghapus karakter tertentu dari teks.

*   from nltk.corpus import stopwords: Mengimpor daftar kata yang umum (stopwords) dari pustaka NLTK. Stopwords adalah kata-kata yang sering muncul dalam bahasa tetapi tidak memiliki makna penting untuk analisis, seperti "dan", "di", "yang", dll.

*   from nltk.tokenize import word_tokenize: Mengimpor fungsi untuk memisahkan (tokenisasi) teks menjadi kata-kata. Tokenisasi adalah langkah awal penting dalam pemrosesan teks.

*   import nltk: Mengimpor pustaka NLTK itu sendiri. Ini adalah pustaka yang sangat kuat untuk  menyediakan berbagai alat dan sumber daya.

*   nltk.download('stopwords'): Mengunduh daftar stopwords yang diperlukan dari repositori NLTK. Ini akan memungkinkan Anda menggunakan daftar kata yang umum dalam bahasa Inggris (atau bahasa lain) untuk membersihkan teks.

*   nltk.download('punkt'): Mengunduh tokenizer yang digunakan untuk memisahkan teks menjadi kata dan kalimat. punkt adalah model untuk memisahkan kalimat dan kata, yang dapat menangani berbagai jenis teks dengan baik.




In [34]:

def preprocess_text(text):
    # Cleaning: Menghapus angka dan tanda baca
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = text.lower()
    words = word_tokenize(text)

    # Stopword removal
    stop_words = set(stopwords.words("indonesian"))
    words = [word for word in words if word not in stop_words]

    return ' '.join(words)

def search_similar_documents(query_text, top_k=5):
    query_text = preprocess_text(query_text)

    # Transformasi query text
    query_tfidf = tfidf_vectorizer.transform([query_text])
    query_svd = svd.transform(query_tfidf)

    # Hitung jarak Euclidean antara query dan dokumen
    distances = euclidean_distances(query_svd, svd_matrix)

    # Ambil indeks dokumen terdekat
    closest_indices = np.argsort(distances[0])[:top_k]
    similarities = distances[0][closest_indices]

    return closest_indices, similarities

# Input teks dan preprocessing
query_text = input("Masukkan teks: ")
processed_query_text = preprocess_text(query_text)

# Melakukan pencarian dengan teks yang telah dipreproses
top_k = 10
indices, distances = search_similar_documents(processed_query_text, top_k)

# Menampilkan hasil pencarian
print("\nHasil pencarian untuk query:", processed_query_text)
for idx, distance in zip(indices, distances):
    print(f"Dokumen ke-{idx + 1}: {documents[idx]}")
    print(f"Jarak Euclidean: {distance:.4f}\n")

Masukkan teks: kendaraan listrik

Hasil pencarian untuk query: kendaraan listrik
Dokumen ke-42: jakarta menteri investasikepala badan koordinasi penanaman modal rosan roeslani bicara potensi indonesia dilirik investor produsen mobil listrik zaman investor asing ragu berinvestasi ekosistem green energy rendah rosan roslani mengungkap potensi energi terbarukan ebt dimilki indonesia mencapai target net zero emmision energi berpotensi indonesia terbarukan nilainya gigawatt potensi bicara potensi berasal tenaga surya angin hydro arus laut biomass panas bumi lainlainnya detikcom leaders forum indonesia hijau inovasi energi sumber daya manusia hotel st regis jakarta selatan selasa senada direktur riset institute for development of economics and finance indef berly martawardaya menyebut pabrikan mundur lantaran indonesia tertinggal negara kawasan asean penggunaan ebt lihat enggan investasi masuk indonesia supply energi terbarukan berly hadir kesempatan manajemen human capital pln yusuf didi se

Nilai jarak ini menunjukkan tingkat kemiripan: semakin kecil nilai jaraknya, semakin relevan dokumen tersebut dengan kata kunci. Dengan jarak 0.8262, dokumen ini dianggap cukup relevan dalam konteks pencarian.

**Cleaning:**

*   re.sub(r'\d+', '', text): Menghapus semua angka dari teks menggunakan ekspresi reguler.

*   text.translate(...): Menghapus semua tanda baca dengan menggunakan metode translate.

*   text.lower(): Mengubah semua huruf dalam teks menjadi huruf kecil untuk menghindari perbedaan antara huruf besar dan kecil saat analisis.

*   word_tokenize(text): Memisahkan teks menjadi kata-kata (token).

**Stopword Removal:**

*   stop_words = set(stopwords.words("indonesian")): Mengambil daftar kata-kata umum yang tidak memiliki makna penting dalam konteks analisis.

*   words = [word for word in words if word not in stop_words]: Menggunakan pemahaman daftar untuk menghapus semua kata yang terdapat dalam daftar stopwords.

*   Return: Fungsi mengembalikan string yang terdiri dari kata-kata yang telah diproses, digabungkan kembali menjadi satu kalimat.

**Preprocessing:**

*   query_text = preprocess_text(query_text): Memanggil fungsi preprocess_text untuk membersihkan dan mempersiapkan teks query sebelum analisis.

**Transformasi:**

*   query_tfidf = tfidf_vectorizer.transform([query_text]): Mengubah teks query yang telah diproses menjadi representasi TF-IDF.

*   query_svd = svd.transform(query_tfidf): Mengubah matriks TF-IDF dari query menjadi bentuk yang direduksi dimensinya menggunakan SVD.

**Menghitung Jarak:**

*   distances = euclidean_distances(query_svd, svd_matrix): Menghitung jarak Euclidean antara representasi SVD dari query dan semua dokumen yang ada dalam matriks SVD.

**Mengambil Dokumen Terdekat:**

*   closest_indices = np.argsort(distances[0])[:top_k]: Mengurutkan jarak dari yang terkecil dan mengambil indeks dari top_k dokumen terdekat.

*   similarities = distances[0][closest_indices]: Mengambil jarak dari dokumen terdekat sesuai dengan indeks yang diperoleh.

**Melakukan Pencarian**

*   top_k = 5 : Variabel ini menetapkan jumlah maksimum dokumen yang ingin ditampilkan sebagai hasil pencarian. Dalam hal ini, hanya 5 dokumen terdekat yang akan diambil berdasarkan kesamaan dengan query.

*   indices, distances = search_similar_documents(processed_query_text, top_k):

*   Fungsi search_similar_documents dipanggil dengan argumen processed_query_text dan top_k.

**Menampilkan Hasil Pencarian**

*   print("\nHasil pencarian untuk query:", processed_query_text):

*   print("\nHasil pencarian untuk query:", processed_query_text):

*   for idx, distance in zip(indices, distances):

*   Menggunakan loop untuk iterasi melalui indices (indeks dokumen terdekat) dan distances (jarak Euclidean untuk setiap dokumen).

*   Fungsi zip digunakan untuk menggabungkan dua list ini sehingga kita dapat mengakses elemen yang berpasangan.

*   print(f"Dokumen ke-{idx + 1}: {documents[idx]}"):

*   Menampilkan informasi tentang dokumen terdekat yang ditemukan, termasuk urutan dokumen (dengan menambahkan 1 pada indeks) dan isi dokumen dari list documents.

*   print(f"Jarak Euclidean: {distance:.4f}\n"):

*   Menampilkan jarak Euclidean antara query dan dokumen tersebut, dengan format hingga 4 angka desimal.



























In [36]:
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import euclidean_distances

# Mengunduh stopwords dan tokenizer
nltk.download('stopwords')
nltk.download('punkt')

# Contoh DataFrame news_data (Anda harus menggantinya dengan DataFrame Anda sendiri)
# news_data = pd.DataFrame({
#     'nomor': [...], # Ensure this column exists in your DataFrame
#     'tanggal': [...],
#     'judul': [...],
#     'kategori': [...],
#     'isi': [...],
#     'stopword_removal': [...]
# })

# ... (rest of the code remains the same) ...

# Menampilkan hasil pencarian
print("\nHasil pencarian untuk query:", processed_query_text)
for idx, distance in zip(indices, distances):
    # Ambil informasi dari news_data, using correct column names if different
    # If 'nomor' column doesn't exist, replace with actual column name
    # For example, if the column with document number is named 'id', use:
    # nomor_berita = news_data['id'].iloc[idx]
    nomor_berita = news_data.iloc[idx].get('nomor', idx + 1)  # Get 'nomor' if exists, else default to idx + 1
    tanggal_berita = news_data['tanggal'].iloc[idx]
    judul_berita = news_data['judul'].iloc[idx]
    kategori_berita = news_data['kategori'].iloc[idx]
    isi_berita = news_data['isi'].iloc[idx]

    # Menampilkan informasi berita
    print(f"Dokumen ke-{nomor_berita}:") # Output using retrieved or default nomor_berita
    print(f"Tanggal: {tanggal_berita}")
    print(f"Judul: {judul_berita}")
    print(f"Kategori: {kategori_berita}")
    print(f"Isi Berita: {isi_berita}")
    print(f"Jarak Euclidean: {distance:.4f}\n")


Hasil pencarian untuk query: kendaraan listrik
Dokumen ke-42:
Tanggal: Rabu, 18 Sep 2024 07:12 WIB
Judul: Wanti-wanti Investasi Mobil Listrik Indonesia Kalah dari Thailand
Kategori: Otomotif
Isi Berita: Jakarta - Menteri Investasi/Kepala Badan Koordinasi Penanaman Modal Rosan Roeslani bicara potensi Indonesia dilirik investor produsen mobil listrik. Tapi zaman sekarang investor asing bisa ragu berinvestasi jika ekosistem green energy yang masih rendah. Rosan Roslani mengungkap potensi energi terbarukan (EBT) yang dimilki Indonesia demi mencapai target net zero emmision. "Kalau dilihat energi yang berpotensi untuk di Indonesia baru terbarukan nilainya 3.677 gigawatt potensi, kita bicara potensi yang di mana berasal tenaga surya, angin, hydro, arus laut, biomass, panas bumi dan lain-lainnya," kata dia dalam detikcom Leaders Forum 'Menuju Indonesia Hijau: Inovasi Energi dan Sumber Daya Manusia,' di Hotel St. Regis, Jakarta Selatan, Selasa (17/9/2024). Senada dengan hal tersebut, Direktur

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
